In [ ]:
day_obs = 20250814
block_name = 'BLOCK-T393_v5'

## Imports

In [ ]:
import galsim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

In [ ]:
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
import os

## Define Functions

In [ ]:
def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

### ConsDB query

In [ ]:
os.environ["no_proxy"] += ",.consdb"
consdb_url = 'http://consdb-pq.consdb:8080/consdb'
cdb_client = ConsDbClient(consdb_url)

In [ ]:
query = f"""
    SELECT
    e.airmass AS airmass,
    e.dimm_seeing AS dimm,
    e.altitude AS elevation,
    e.azimuth AS azimuth,
    e.exposure_id AS visit_id,
    e.physical_filter as band,
    e.day_obs AS day_obs,
    e.exp_midpt AS time,
    e.dimm_seeing AS seeing,
    e.science_program AS science_program,
    e.group_id AS group_id,
    e.seq_num AS seq,
    ccdvisit1_quicklook.psf_sigma,
    ccdvisit1_quicklook.z4,
    ccdvisit1_quicklook.z5,
    ccdvisit1_quicklook.z6,
    ccdvisit1_quicklook.z7,
    ccdvisit1_quicklook.z8,
    ccdvisit1_quicklook.z9,
    ccdvisit1_quicklook.z10,
    ccdvisit1_quicklook.z11,
    ccdvisit1_quicklook.z12,
    ccdvisit1_quicklook.z13,
    ccdvisit1_quicklook.z14,
    ccdvisit1_quicklook.z15,
    ccdvisit1_quicklook.z16,
    ccdvisit1_quicklook.z17,
    ccdvisit1_quicklook.z18,
    ccdvisit1_quicklook.z19,
    ccdvisit1_quicklook.z20,
    ccdvisit1_quicklook.z21,
    ccdvisit1_quicklook.z22,
    ccdvisit1_quicklook.z23,
    ccdvisit1_quicklook.z24,
    ccdvisit1_quicklook.z25,
    ccdvisit1_quicklook.z26,
    ccdvisit1_quicklook.z27,
    ccdvisit1_quicklook.z28,
    ccdvisit1.detector as detector,
    q.psf_sigma_median AS psf_fwhm,
    q.psf_sigma_min AS psf_fwhm_min,
    q.psf_sigma_max AS psf_fwhm_max,
    e.obs_end,
    e.obs_start
    FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1 AS ccdvisit1,
    cdb_lsstcam.visit1 AS visit1,
    cdb_lsstcam.visit1_quicklook AS q,
    cdb_lsstcam.exposure AS e
    WHERE
    ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
    AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id = visit1.visit_id
    AND ccdvisit1.visit_id = q.visit_id
    AND ccdvisit1.visit_id = e.exposure_id
    AND (e.img_type = 'science' or e.img_type = 'acq')
    AND e.day_obs = {day_obs}
    AND e.science_program = '{block_name}'
    AND e.airmass > 0
    AND e.band != 'none'
"""
cdb_table = cdb_client.query(query).to_pandas()

# Convert PSF sigma to FWHM
sig2fwhm = 2 * np.sqrt(2 * np.log(2))
pixel_scale = 0.2  # arcsec / pixel
cdb_table["psf_fwhm"] = cdb_table["psf_fwhm"] * sig2fwhm * pixel_scale

cdb_table["fwhm_zenith_500nm"] = [
    fwhm
    * getAirmassSeeingCorrection(airmass)
    * getBandpassSeeingCorrection(band)
    for fwhm, band, airmass in zip(
        cdb_table["psf_fwhm"], cdb_table["band"], cdb_table["airmass"]
    )
]

zernike_columns = [f"z{i}" for i in range(4, 29)]
cdb_table["zernikes"] = cdb_table[zernike_columns].apply(
    lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
)
cdb_table["zernikes_fwhm"] = cdb_table["zernikes"].apply(
    convertZernikesToPsfWidth
)
cdb_table["aos_fwhm"] = 1.06 * np.log(
    1
    + np.sqrt(
        np.sum(np.square(np.vstack(cdb_table["zernikes_fwhm"].values)), axis=1)
    )
)

In [ ]:
group_names = np.unique([x.split('#')[0] for x in cdb_table['group_id'].unique()])

In [ ]:
group_seq = dict()
for group_info in zip(cdb_table[['group_id', 'seq']].values):
    group_name, seq_num = group_info[0]
    if (group_name not in group_seq.keys()) and (group_name in group_names):
        group_seq[group_name] = [seq_num]
    elif group_name in group_seq.keys():
        if seq_num not in group_seq[group_name]:
            group_seq[group_name].append(seq_num)
    else:
        continue

In [ ]:
cdb_sub_table = cdb_table[['elevation', 'dimm', 'aos_fwhm', 'fwhm_zenith_500nm', 'band', 'science_program', 'group_id', 'seq']]
cdb_sub_table = cdb_sub_table.groupby('seq').agg({'elevation': 'mean', 'dimm': 'mean', 'aos_fwhm': 'mean', 'fwhm_zenith_500nm': 'mean', 'band': 'first', 'science_program': 'first', 'group_id': 'first'})

In [ ]:
aos_fwhm = []
fwhm_zenith_500nm = []
dimm = []
elevation = []
for group_name in group_names:
    aos = []
    fwhm = []
    d = []
    e = []
    for seq_on in group_seq[group_name]:
        seq_info = cdb_sub_table.loc[seq_on]
        aos.append(seq_info['aos_fwhm'])
        d.append(seq_info['dimm'])
        fwhm.append(seq_info['fwhm_zenith_500nm'])
        e.append(seq_info['elevation'])
    aos_fwhm.append(aos)
    fwhm_zenith_500nm.append(fwhm)
    dimm.append(d)
    elevation.append(e)

## Create plots

In [ ]:
plotted_group_idx = list()
for idx, group_name in enumerate(group_names):
    iter_vals = 1 + (np.array(group_seq[group_name]) - group_seq[group_name][0]) / 2
    if len(iter_vals) < 5:
        continue
    plt.plot(iter_vals, aos_fwhm[idx], '-o', label=f'Starting Elev: {elevation[idx][0]:.2f}')
    plotted_group_idx.append(idx)
plt.legend()
plt.xticks(np.arange(1,11))
plt.ylabel('AOS FWHM (arcsec)')
plt.xlabel('Iteration')
plt.title(f'Closed Loop Tests: {block_name} on {day_obs}')

In [ ]:
num_subplot_cols = 2
num_subplot_rows = int(np.ceil(len(plotted_group_idx)/num_subplot_cols))
fig, axes = plt.subplots(num_subplot_rows, num_subplot_cols, figsize=(6*num_subplot_cols, 4*num_subplot_rows))

n_plots = len(plotted_group_idx)
# Flatten axes array to make indexing easier
axes = axes.flatten() if n_plots > 1 else [axes]

# Plot each element
for plot_idx, data_idx in enumerate(plotted_group_idx):
    x_vals = 1 + (np.array(group_seq[group_names[data_idx]]) - group_seq[group_names[data_idx]][0]) / 2
    axes[plot_idx].plot(x_vals, aos_fwhm[data_idx], '-o', label='AOS FWHM')
    axes[plot_idx].plot(x_vals, dimm[data_idx], '-o', label='DIMM')
    axes[plot_idx].plot(x_vals, fwhm_zenith_500nm[data_idx], '-o', label='fwhn_zenith_500nm')
    axes[plot_idx].set_title(f'Group ID: {group_names[0]}, Starting elev: {elevation[data_idx][0]:.2f}')
    axes[plot_idx].set_xticks(np.arange(1, 11))
    axes[plot_idx].legend()
    axes[plot_idx].set_ylabel('FWHM (arcsec)')
    axes[plot_idx].set_xlabel('Iteration')

# Hide empty subplots if any
for i in range(n_plots, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()